# Learning 4: Tools Basics

**Goal**: Create tools that LLMs can use to perform actions

## What You'll Learn
- What tools are and why they matter
- Creating tools with the `@tool` decorator
- Binding tools to an LLM
- Understanding tool calls

In [5]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

llm = ChatOpenAI(model="gpt-4o-mini")
print("Setup complete!")

Setup complete!


## What are Tools?

Tools are functions that an LLM can decide to call. They extend the LLM's capabilities beyond just generating text:

- **Calculator**: Do math calculations
- **Search**: Look up information
- **Database**: Query or update data
- **API calls**: Interact with external services

## Creating Tools with @tool Decorator

In [6]:
@tool
def add(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b

# See tool information
print("Tool name:", add.name)
print("Tool description:", add.description)
print("Tool schema:", add.args_schema.schema())

Tool name: add
Tool description: Add two numbers together.
Tool schema: {'description': 'Add two numbers together.', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'add', 'type': 'object'}


/var/folders/gy/v6xz068x1f9_ybg3xp_mq5lr0000gn/T/ipykernel_4053/3681212514.py:14: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print("Tool schema:", add.args_schema.schema())


In [7]:
# Call the tool directly
result = add.invoke({"a": 5, "b": 3})
print("5 + 3 =", result)

5 + 3 = 8


## More Realistic Tool Examples

In [8]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # In a real app, this would call a weather API
    weather_data = {
        "new york": "Sunny, 72°F",
        "london": "Cloudy, 58°F",
        "tokyo": "Rainy, 65°F"
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}")

@tool
def search_database(query: str) -> str:
    """Search the product database."""
    # Simulated database search
    products = {
        "laptop": "MacBook Pro - $1999",
        "phone": "iPhone 15 - $999",
        "headphones": "AirPods Pro - $249"
    }
    for key, value in products.items():
        if key in query.lower():
            return value
    return "No products found"

print(get_weather.invoke({"city": "Tokyo"}))
print(search_database.invoke({"query": "looking for a laptop"}))

Rainy, 65°F
MacBook Pro - $1999


## Binding Tools to an LLM

To let the LLM use tools, we "bind" them to the model.

In [ ]:
# Create tools list
tools = [add, multiply, get_weather]

# Bind tools to LLM
llm_with_tools = llm.bind_tools(tools)

print("LLM now has access to:", [t.name for t in tools])

## Understanding Tool Calls

When the LLM decides to use a tool, it returns a "tool call" instead of a regular response.

In [ ]:
# Ask a question that requires a tool
response = llm_with_tools.invoke("What is 15 + 27?")

print("Response content:", response.content)
print("Tool calls:", response.tool_calls)

In [ ]:
# Look at the tool call details
if response.tool_calls:
    tool_call = response.tool_calls[0]
    print("Tool name:", tool_call["name"])
    print("Tool args:", tool_call["args"])
    print("Tool ID:", tool_call["id"])

## Executing Tool Calls

After the LLM requests a tool call, we need to:
1. Execute the tool
2. Send the result back to the LLM

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

# Step 1: Get the tool call
response = llm_with_tools.invoke("What's the weather in London?")
print("LLM wants to call:", response.tool_calls)

# Step 2: Execute the tool
tool_call = response.tool_calls[0]
tool_result = get_weather.invoke(tool_call["args"])
print("Tool result:", tool_result)

# Step 3: Send result back to LLM
messages = [
    HumanMessage(content="What's the weather in London?"),
    response,  # The AI's response with tool call
    ToolMessage(content=tool_result, tool_call_id=tool_call["id"])
]

final_response = llm_with_tools.invoke(messages)
print("Final answer:", final_response.content)

## Multiple Tool Calls

Sometimes the LLM needs to call multiple tools.

In [ ]:
response = llm_with_tools.invoke(
    "What is 5 + 3, and also what is 4 * 6?"
)

print(f"Number of tool calls: {len(response.tool_calls)}")
for tc in response.tool_calls:
    print(f"  - {tc['name']}: {tc['args']}")

## Tool with Complex Input

In [ ]:
from typing import List

@tool
def create_meeting(
    title: str,
    attendees: List[str],
    duration_minutes: int = 30
) -> str:
    """Create a new meeting with the given details."""
    return f"Meeting '{title}' created with {len(attendees)} attendees for {duration_minutes} minutes"

# Bind and test
llm_with_meeting = llm.bind_tools([create_meeting])
response = llm_with_meeting.invoke(
    "Schedule a meeting called 'Project Review' with Alice and Bob for 1 hour"
)

print("Tool call:", response.tool_calls[0])

## Exercise: Create Your Own Tools

1. Create a tool that converts temperatures (Celsius to Fahrenheit)
2. Create a tool that looks up user information
3. Bind them and test with the LLM

In [ ]:
# Your code here!

# @tool dectorator , tool calls, bind_tools

## Key Takeaways

1. `@tool` decorator creates tools from functions
2. The docstring becomes the tool description (important for LLM!)
3. Type hints define the expected parameters
4. `.bind_tools()` gives the LLM access to tools
5. Tool calls must be executed and results sent back to LLM

**Next**: Learning 5 - First Graph (LangGraph!)